# Ingest Weather Documents -> Vector Embeddings (Lakebase)

Reads `weather_documents` rows with no existing `weather_embeddings` row for the current model, chunks `narrative_text`, embeds each chunk with `sentence-transformers`, and writes vectors into `weather_embeddings` via psycopg2 (`%s::vector` cast, no Spark JDBC).

Uses the same `LAKEBASE_URL` secret / connection setup as `app.py` and `lakebase.py`.

In [0]:
#%pip uninstall -y psycopg2 psycopg2-binary

In [0]:
%pip install -q 'databricks-sdk>=0.30.0' 'sqlalchemy>=2.0.30'

In [0]:
%pip install -q sentence-transformers

In [0]:
dbutils.library.restartPython()

## Import libraries and set config.

In [0]:
import hashlib
import os

from psycopg2.extras import execute_values
from sentence_transformers import SentenceTransformer
import lakebase

In [0]:
EMBEDDING_MODEL_NAME = os.environ.get(
    "WEATHER_EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
)
# Most NWS narrative text is short; chunking mainly matters for combined alert description + instruction text which may be large.
CHUNK_SIZE = int(os.environ.get("WEATHER_CHUNK_SIZE", 800))
CHUNK_OVERLAP = int(os.environ.get("WEATHER_CHUNK_OVERLAP", 100))
ENCODE_BATCH_SIZE = int(os.environ.get("WEATHER_EMBED_BATCH_SIZE", 32))

print(f"Using model {EMBEDDING_MODEL_NAME!r}, chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")

## Fetch unembedded documents

Documents with no existing `weather_embeddings` row under the current model. This is an optimization to skip already-processed documents on repeat runs, not a strict correctness guarantee - if a document's `narrative_text` changes after being embedded (same document id, e.g. a re-issued forecast period), it won't be picked up again here since it already has at least one embedding row. Re-embedding on content change is limitation (updated in `README_WEATHER.md`).

In [0]:
def fetch_unembedded_documents() -> list[dict]:
    return lakebase.run_query(
        """
        SELECT d.id, d.narrative_text
        FROM weather_documents d
        LEFT JOIN weather_embeddings e
            ON e.document_id = d.id AND e.model_name = %s
        WHERE e.id IS NULL
          AND d.narrative_text IS NOT NULL
          AND TRIM(d.narrative_text) != ''
        """,
        (EMBEDDING_MODEL_NAME,),
    )


documents = fetch_unembedded_documents()
print(f"Found {len(documents)} document(s) to embed.")

## Chunk narrative text

Sliding-window chunking - most inputs are shorter than `CHUNK_SIZE` and produce exactly one chunk.

In [0]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    text = text.strip()
    if not text:
        return []

    chunks = []
    step = max(chunk_size - overlap, 1)
    for start in range(0, len(text), step):
        chunk = text[start : start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(text):
            break
    return chunks


chunk_rows = []
for doc in documents:
    for chunk_index, chunk in enumerate(chunk_text(doc["narrative_text"])):
        chunk_rows.append((doc["id"], chunk_index, chunk))

print(f"Produced {len(chunk_rows)} chunk(s).")

## Compute embeddings

Loads the model once and encodes in batches of `ENCODE_BATCH_SIZE`.

In [0]:
if chunk_rows:
    print(f"Loading embedding model {EMBEDDING_MODEL_NAME!r}...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    print(f"Embedding {len(chunk_rows)} chunk(s) in batches of {ENCODE_BATCH_SIZE}...")
    chunk_texts = [row[2] for row in chunk_rows]
    embeddings = []
    for i in range(0, len(chunk_texts), ENCODE_BATCH_SIZE):
        batch = chunk_texts[i : i + ENCODE_BATCH_SIZE]
        vectors = model.encode(batch, show_progress_bar=False)
        embeddings.extend(vectors.tolist())
        print(f"  Embedded {min(i + ENCODE_BATCH_SIZE, len(chunk_texts))}/{len(chunk_texts)} chunks")
else:
    embeddings = []
    print("No chunks to embed.")

## Write embeddings into Lakebase

Cast to `vector` directly in the `INSERT` via `%s::vector` - no intermediate array + separate cast step needed.

In [0]:
def to_vector_literal(embedding) -> str:
    return "[" + ",".join(repr(float(x)) for x in embedding) + "]"


if embeddings:
    insert_rows = [
        (
            hashlib.sha256(f"{document_id}:{chunk_index}".encode("utf-8")).hexdigest(),
            document_id,
            chunk_index,
            chunk_text_value,
            to_vector_literal(embedding),
            EMBEDDING_MODEL_NAME,
        )
        for (document_id, chunk_index, chunk_text_value), embedding in zip(chunk_rows, embeddings)
    ]

    print(f"Writing {len(insert_rows)} embedding row(s) into weather_embeddings...")
    with lakebase.get_connection() as conn:
        with conn.cursor() as cur:
            execute_values(
                cur,
                """
                INSERT INTO weather_embeddings (
                    id, document_id, chunk_index, chunk_text, embedding, model_name, created_at
                ) VALUES %s
                ON CONFLICT (id) DO UPDATE
                    SET chunk_text = EXCLUDED.chunk_text,
                        embedding = EXCLUDED.embedding,
                        model_name = EXCLUDED.model_name,
                        created_at = EXCLUDED.created_at
                """,
                insert_rows,
                template="(%s, %s, %s, %s, %s::vector, %s, now())",
                page_size=100,
            )
        conn.commit()

    print(f"Done. Inserted/updated {len(insert_rows)} embedding row(s).")
else:
    print("Nothing to write.")